# Partition Exploration

Interactive analysis of $p = 2^m + q^n$ across the full 10B-prime iceberg catalog.

**Stack**: Polars lazy scans via `pl.scan_iceberg` with predicate pushdown through
manifest statistics. Altair + VegaFusion for server-side chart aggregation.
SageMath for number-theoretic computations (polynomial rings, finite fields).

**Design principle**: polars expressions are first-class objects. Define once,
reuse across `filter`, `select`, `agg`, `with_columns`, `over`. Read the
`.explain()` plan to understand what polars will do; use `collect(engine="streaming")`
for full-table passes; use `collect_chunked` when a single plan would exceed u32.

In [2]:
import sys, os

_project_root = os.path.dirname(os.path.abspath(os.getcwd()))
os.chdir(_project_root)
sys.path.insert(0, os.path.join(_project_root, "src"))

import altair as alt
import polars as pl

alt.data_transformers.enable("vegafusion")

from funbuns.iceberg_query import (
    open_catalog,
    scan_primes,
    scan_decompositions,
    total_rows,
    max_p,
    k_distribution,
    chunks_by_commit_seq,
    collect_chunked,
)

catalog = open_catalog()
primes_lf = scan_primes(catalog)
decomp_lf = scan_decompositions(catalog)

# Metadata-only ground truth — zero parquet reads.
n_primes  = total_rows(catalog, "primes")
n_decomp  = total_rows(catalog, "decompositions")
largest_p = max_p(catalog)

print(f"primes:     {n_primes:>15,}")
print(f"decomp:     {n_decomp:>15,}")
print(f"largest p:  {largest_p:>15,}")

primes:      10,099,850,000
decomp:      19,011,566,647
largest p:  254,824,597,673


## Expressions are first-class objects

Polars expressions are reusable Python values. Define once, drop into
`filter`, `select`, `agg`, `with_columns`. The optimizer fuses them into
a single plan. This is the core difference from pandas — not the syntax,
but that an expression is a composable, inspectable object.

In [4]:
# Named expressions — build once, reuse everywhere.
is_obstructed = pl.col("k") == 0
has_decomps   = pl.col("k") > 0

# Same `is_obstructed` used in:
#   - a conditional aggregate (boolean .sum() counts True)
#   - a per-aggregation filter (pl.col().filter() inside agg)
#   - an expression key for group_by (no pre-computed column needed)
summary = (
    primes_lf
    .group_by((pl.col("p") // 10**10).alias("decade"))
    .agg([
        pl.len().alias("n_primes"),
        is_obstructed.sum().alias("n_obstructed"),
        pl.col("k").filter(has_decomps).mean().alias("mean_k_nonobs"),
        pl.col("k").filter(has_decomps).max().alias("k_max"),
    ])
    .sort("decade")
)

# Read the plan before collecting.
print(summary.explain(optimized=True))
print()
#summary.collect(engine="streaming")

SORT BY [col("decade")]
  AGGREGATE[maintain_order: false]
    [len().alias("n_primes"), [(col("k")) == (0)].sum().alias("n_obstructed"), col("k").filter(col("__POLARS_CSER_0x195dfd8952b425af")).mean().alias("mean_k_nonobs"), col("k").filter(col("__POLARS_CSER_0x195dfd8952b425af")).max().alias("k_max")] BY [[(col("p")) // (10000000000)].alias("decade")]
    FROM
     WITH_COLUMNS:
     [[(col("k")) > (0)].alias("__POLARS_CSER_0x195dfd8952b425af")] 
      Parquet SCAN [/media/extssd/research/dioph.pp/data/iceberg/warehouse/funbuns/primes/data/commit_seq=504/primes_b000504_000.parquet, ... 504 other sources]
      PROJECT 2/3 COLUMNS



## Window functions with `over()`

`over()` runs a computation partitioned by group **without collapsing rows**.
Here: for each decomposition template $(q, n)$, rank primes by `p` and
extract the first prime that realizes it. The rank decorates every row;
filtering on `rank == 1` gives min-p-per-template while preserving all
columns from the original row.

In [ ]:
# First prime per (q_k, n_k) template.
#
# Earlier draft used rank().over(["q_k","n_k"]).filter(rank==1). That
# pattern keeps 19B rows (over() doesn't collapse) and rank isn't in
# the streaming engine's op set — it either OOMs or silently falls back
# to in-memory and OOMs there. group_by+min is streamable and produces
# exactly one row per (q_k, n_k) group.
first_per_template_lf = (
    decomp_lf
    .group_by(["q_k", "n_k"])
    .agg(pl.col("p").min().alias("p"))
    .sort(["q_k", "n_k"])
)

print(first_per_template_lf.explain(optimized=True))
print()

first_per_template = first_per_template_lf.collect(engine="streaming")
print(f"{first_per_template.height:,} distinct (q, n) templates")
first_per_template.head(20)

## Multi-solution $(p, q)$ pairs — list aggregations + chunked collect

Known result: across the full dataset, only $(p{=}11, q{=}3)$ admits two
distinct $(m, n)$ solutions per $(p, q)$ pair (S-unit equation bound).

`group_by` with list-typed output columns captures the solution set per
group. `collect_chunked` walks the 19B-row decomposition table one
`commit_seq` partition at a time — each chunk fits under polars' u32 row
counter, and the `reduce` step merges partial results across chunks.

In [ ]:
multi = collect_chunked(
    catalog, "decompositions",
    build=lambda lf: (
        lf.group_by(["p", "q_k"])
        .agg([
            pl.len().alias("n_sol"),
            pl.col("m_k").alias("ms"),
            pl.col("n_k").alias("ns"),
        ])
        .filter(pl.col("n_sol") > 1)
    ),
    progress=True,
)

# Expected: exactly one row — p=11, q=3, ms=[1,3], ns=[2,1]
multi

## The q-chain DAG as a self-join

Each decomposition row is a directed edge $p \to q_k$ in the
"built from" graph. Since $q_k < p$ always, this graph is a DAG.
Multi-hop chains are self-joins on `decomp`: the right side's `p`
matches the left side's `q_k`.

**In-degree** of a prime $q$ = how many larger primes use it as a base.
**Sinks** = primes with out-degree 0 (obstructed).
**Sources** = primes with in-degree 0 (no larger prime decomposes through them).

In [ ]:
# 2-hop chains: p -> mid -> q2 where both p and mid decompose.
#
# A naive self-join of decomp_lf on itself is two 19B-row scans. Polars'
# predicate pushdown pushes a filter into the scan of the branch where
# it's defined, but it does NOT propagate across join branches — a
# filter on left.p is not automatically mirrored onto right.p. We push
# both filters manually so each side prunes iceberg files independently.
#
# Constraint: right.p = left.q_k (renamed to `mid`). Values of q_k
# observed in the left when left.p < 1000 are small (all odd primes
# that appear as bases for some p < 1000) — bounded above by ~1000 as
# well since q^n < p. So right.p < 1000 is a valid superset and keeps
# the right-side scan pruned to low-p-range iceberg files.
P_CAP = 1000

left = (
    decomp_lf
    .filter(pl.col("p") < P_CAP)
    .select([
        pl.col("p"),
        pl.col("m_k").alias("m1"),
        pl.col("n_k").alias("n1"),
        pl.col("q_k").alias("mid"),
    ])
)
right = (
    decomp_lf
    .filter(pl.col("p") < P_CAP)
    .select([
        pl.col("p").alias("mid"),
        pl.col("m_k").alias("m2"),
        pl.col("n_k").alias("n2"),
        pl.col("q_k").alias("q2"),
    ])
)

joined = left.join(right, on="mid", how="inner").sort(["p", "mid", "q2"])
print(joined.explain(optimized=True))
print()

hop2 = joined.collect(engine="streaming")
print(f"{hop2.height} two-hop chains below p={P_CAP}")
hop2.head(15)

In [ ]:
# In-degree distribution: how many primes use each q as a base?
# This is a full-table aggregate but the group_by output is small
# (one row per distinct q_k value).
in_deg = collect_chunked(
    catalog, "decompositions",
    build=lambda lf: lf.group_by("q_k").agg(pl.len().alias("in_deg")),
    reduce=lambda df: (
        df.group_by("q_k")
        .agg(pl.col("in_deg").sum())
        .sort("in_deg", descending=True)
    ),
    progress=True,
)

print(f"{in_deg.height:,} distinct q values as decomposition bases")
in_deg.head(20)

## Covering-system residue classifier

Obstructed primes ($k=0$) tend to cluster in specific residue classes
modulo small moduli — Erdős covering systems with backbone $\{3, 5, 7\}$.
The `when/then/otherwise` chain is the polars idiom for conditional
labeling: no `.loc[]` indexing, no separate mask arrays.

In [ ]:
# Illustrative classifier — the full 22-class mod-255255 version lives
# in scripts/. This shows the when/then pattern.
classified = (
    primes_lf
    .filter(is_obstructed)
    .with_columns([
        (pl.col("p") % 3).alias("r3"),
        (pl.col("p") % 5).alias("r5"),
        (pl.col("p") % 7).alias("r7"),
        (pl.col("p") % 8).alias("r8"),
    ])
    .with_columns(
        pl.when(pl.col("r3") == 2)
          .then(pl.lit("3-blocked"))
        .when((pl.col("r3") == 1) & (pl.col("r5") == 3))
          .then(pl.lit("3,5-blocked"))
        .when((pl.col("r3") == 1) & (pl.col("r5").is_in([1, 2, 4])) & (pl.col("r7") == 4))
          .then(pl.lit("3,5,7-blocked"))
        .otherwise(pl.lit("other"))
        .alias("cover_class")
    )
    .group_by("cover_class")
    .agg([
        pl.len().alias("n"),
        pl.col("p").min().alias("p_min"),
        pl.col("p").max().alias("p_max"),
    ])
    .sort("n", descending=True)
    .collect(engine="streaming")
)
classified

## k-distribution: two paths

**Metadata path**: `k_distribution(catalog)` sums the per-file
`funbuns.k_histogram` KV blocks from parquet footers. Zero data reads.

**Data path**: `collect_chunked` with a `group_by("k")` aggregate.
Full scan, chunked to stay under u32. The two should agree exactly —
any discrepancy means the writer's footer stamping has drifted.

In [ ]:
# Metadata path (instant).
k_meta = k_distribution(catalog)
print("From footer KV metadata:")
display(k_meta)

# Data path (full scan).
k_data = collect_chunked(
    catalog, "primes",
    build=lambda lf: lf.group_by("k").agg(pl.len().alias("count")),
    reduce=lambda df: df.group_by("k").agg(pl.col("count").sum()).sort("k"),
    progress=True,
)
print("\nFrom data scan:")
display(k_data)

# Cross-check.
assert k_meta.sort("k")["count"].to_list() == k_data["count"].to_list(), (
    "k-distribution mismatch between footer KV and data scan"
)
print("\n✓ footer KV and data scan agree")

In [ ]:
alt.Chart(k_meta).transform_filter(
    alt.datum.count > 0
).mark_bar().encode(
    x=alt.X("k:O", title="decomposition count k"),
    y=alt.Y("count:Q", title="number of primes",
            stack=None,
            scale=alt.Scale(type="log")),
    tooltip=["k", "count"]
).properties(width=500, title="k-distribution (footer metadata)")

## k-distribution across p-range blocks

How does the decomposition count evolve as primes get larger?
Bin by fixed p-width (blocks are unequal in prime count because
density decreases as $p$ grows). The expression key
`(pl.col("p") // BLOCK_WIDTH)` goes straight into `group_by` —
no pre-computed column.

In [ ]:
BLOCK_WIDTH = 10_000_000_000  # 10^10

k_by_block = (
    primes_lf
    .group_by([
        (pl.col("p") // BLOCK_WIDTH).alias("block"),
        "k",
    ])
    .agg([
        pl.len().alias("count"),
        pl.col("p").min().alias("p_min"),
        pl.col("p").max().alias("p_max"),
    ])
    .sort(["block", "k"])
    .collect(engine="streaming")
)

# Per-block summary: mean k of non-obstructed primes.
block_summary = (
    k_by_block
    .with_columns((pl.col("k") * pl.col("count")).alias("k_weighted"))
    .group_by("block")
    .agg([
        pl.col("k_weighted").sum().alias("sum_k"),
        pl.col("count").sum().alias("n_primes"),
        pl.col("p_min").min().alias("p_min"),
        pl.col("p_max").max().alias("p_max"),
    ])
    .with_columns((pl.col("sum_k") / pl.col("n_primes")).alias("mean_k"))
    .sort("block")
)
display(block_summary)

alt.Chart(k_by_block).mark_rect().encode(
    x=alt.X("block:O", title=f"p-range block (width = {BLOCK_WIDTH:.0e})"),
    y=alt.Y("k:O", title="k", sort="descending"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="viridis", type="log"),
                    title="primes"),
    tooltip=["block", "k", "count", "p_min", "p_max"]
).properties(width=600, height=350,
             title="k-distribution evolution across p-range")

## q-frequency across prime range

How does the frequency of small base primes change as $p$ grows?

In [ ]:
BIN_WIDTH = 10_000_000_000  # 10^10

q_by_range = collect_chunked(
    catalog, "decompositions",
    build=lambda lf: (
        lf.filter(pl.col("q_k") <= 20)
        .group_by([
            (pl.col("p") // BIN_WIDTH).alias("p_bin"),
            "q_k",
        ])
        .agg(pl.len().alias("count"))
    ),
    reduce=lambda df: (
        df.group_by(["p_bin", "q_k"])
        .agg(pl.col("count").sum())
        .sort(["p_bin", "q_k"])
    ),
    progress=True,
)

alt.Chart(q_by_range).mark_line().encode(
    x=alt.X("p_bin:Q", title="p-range block", axis=alt.Axis(format="~s")),
    y=alt.Y("count:Q", title="decompositions"),
    color=alt.Color("q_k:N", title="q"),
    tooltip=["p_bin", "q_k", "count"]
).properties(width=700, height=350, title="q frequency by p-range (q ≤ 20)")

## Polynomial template expansion

Each decomposition $p = 2^m + q^n$ defines a template $f_{m,n}(X) = 2^m + X^n$.
If $q$ itself decomposes as $q = 2^{m'} + q'^{n'}$, substituting gives a chain:

$$p = f_{m,n}(f_{m',n'}(q')) = 2^m + (2^{m'} + q'^{n'})^n$$

Since $q < p$ always, the decomposition graph is a DAG. Processing primes
in ascending order is topological: each subtree is expanded exactly once.

This section uses SageMath for polynomial composition and $\mathbb{F}_p$ evaluation.

In [ ]:
import itertools
from collections import defaultdict
from sage.all import ZZ, PolynomialRing, GF, Mod

P_LIMIT = 149  # first totally obstructed prime

# Pull decomposition data from iceberg — tiny scan.
primes_up_to = (
    primes_lf
    .filter(pl.col("p") <= P_LIMIT)
    .select(["p", "k"])
    .collect(engine="streaming")
)
k_by_p = dict(primes_up_to.iter_rows())

decomps_up_to = (
    decomp_lf
    .filter((pl.col("p") <= P_LIMIT) & (pl.col("q_k") > 0))
    .select(["p", "m_k", "q_k", "n_k"])
    .collect(engine="streaming")
)

decomp_map = defaultdict(list)
for row in decomps_up_to.iter_rows(named=True):
    decomp_map[row["p"]].append((row["m_k"], row["q_k"], row["n_k"]))

obstructed = sorted(p for p, k in k_by_p.items() if k == 0)
print(f"Primes: {len(k_by_p)} ({len(decomp_map)} with decompositions)")
print(f"Obstructed (k=0): {obstructed}")

# Polynomial ring
R = PolynomialRing(ZZ, "q")
q = R.gen()

def template(m, n):
    return ZZ(2)**m + q**n

templates = sorted(set(
    (m, n) for m, _, n in itertools.chain.from_iterable(decomp_map.values())
))
template_polys = {(m, n): template(m, n) for m, n in templates}
poly_to_template = {f: (m, n) for (m, n), f in template_polys.items()}

print(f"\n{len(templates)} distinct templates:")
for m, n in templates:
    print(f"  f_{{{m},{n}}} = {template_polys[(m,n)]}")

In [ ]:
# Bottom-up memoized chain expansion.
def build_chain_memo(decomp_map):
    memo = {}
    for p in sorted(decomp_map):
        chains = []
        for m, q_val, n in decomp_map[p]:
            f = template(m, n)
            sub = memo.get(q_val, [(q_val, [], q)])
            chains.extend(
                (tq, [(m, n, q_val)] + st, f(sp))
                for tq, st, sp in sub
            )
        memo[p] = chains
    return memo

chain_memo = build_chain_memo(decomp_map)

# Flatten into a polars DataFrame + polynomial lookup.
poly_lookup = {}
flat_rows = []
for p, chains in sorted(chain_memo.items()):
    for tq, st, poly in chains:
        ps = str(poly)
        flat_rows.append((p, k_by_p[p], tq, len(st), ps, poly.degree()))
        poly_lookup[ps] = poly

chain_df = pl.DataFrame(
    flat_rows,
    schema=["source_p", "k", "terminal_q", "depth", "poly", "degree"],
    orient="row",
)

print(f"Total chains: {len(chain_df)}")
print(f"Unique polynomials: {len(poly_lookup)}")

# Polars analytics on the chain catalog.
print("\nChains by depth:")
display(chain_df.group_by("depth").len().sort("depth"))

print("\nTerminal q distribution:")
display(
    chain_df
    .group_by("terminal_q")
    .agg([
        pl.len().alias("chains"),
        pl.col("source_p").n_unique().alias("primes"),
    ])
    .sort("terminal_q")
    .with_columns(
        pl.when(pl.col("terminal_q").is_in(obstructed))
        .then(pl.lit("OBSTRUCTED"))
        .when(pl.col("terminal_q") > P_LIMIT)
        .then(pl.lit("OUT OF RANGE"))
        .otherwise(pl.lit(""))
        .alias("status")
    )
)

# Polynomials shared by multiple source primes.
shared = (
    chain_df
    .group_by("poly")
    .agg([
        pl.col("source_p").unique().sort().alias("primes"),
        pl.col("source_p").n_unique().alias("n_primes"),
    ])
    .filter(pl.col("n_primes") > 1)
    .sort("n_primes", descending=True)
)
print("\nPolynomials shared by multiple primes:")
display(shared)

### Obstructed primes: $\mathbb{F}_p$ root structure

No single template $f_{m,n}$ has a root in $\mathbb{F}_p$ for obstructed $p$.
But composed polynomials from multi-hop chains might — revealing algebraic
structure that single-step decomposition misses.

In [ ]:
def eval_polys_at(target_p, poly_lookup):
    """Evaluate all polynomials over GF(target_p). Return those with roots."""
    Fp = GF(target_p)
    Rp = PolynomialRing(Fp, "x")
    return [
        (ps, [ZZ(r) for r, _ in Rp(poly).roots()])
        for ps, poly in poly_lookup.items()
        if Rp(poly).roots()
    ]

print("Obstructed primes: composed polynomial roots in GF(p)")
print("=" * 60)
for obs_p in obstructed:
    hits = eval_polys_at(obs_p, poly_lookup)
    print(f"\np = {obs_p}: {len(hits)}/{len(poly_lookup)} have roots")
    for ps, roots in hits[:5]:
        print(f"  {ps}  →  {[int(r) for r in roots]}")
    if len(hits) > 5:
        print(f"  ... and {len(hits) - 5} more")

## Scratch space

Available bindings:

- `primes_lf`, `decomp_lf` — lazy frames over iceberg tables
- `catalog` — `SqlCatalog` handle
- `is_obstructed`, `has_decomps` — reusable expressions
- `collect_chunked(catalog, table, build, *, reduce)` — full-table aggregate, chunked to avoid u32
- `k_distribution(catalog)` — instant k-histogram from footer KV
- `chain_memo`, `poly_lookup`, `chain_df` — polynomial chain catalog

Use `.explain()` to understand any query before collecting.
Use `collect(engine="streaming")` for single-plan full-table passes.
Use `collect_chunked` when the full table exceeds u32.